# LDA vs PCA: Supervised Meets Unsupervised Dimensionality Reduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/lda_vs_pca.ipynb)

This notebook accompanies the blog post at [sesen.ai](https://sesen.ai/blog/lda-vs-pca-supervised-unsupervised-dimensionality-reduction).

We compare PCA (unsupervised) and Fisher's LDA (supervised) for dimensionality reduction on the Iris dataset, then explore an adversarial example where PCA completely fails.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

## 1. Load and Prepare the Iris Dataset

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names
feature_names = iris.feature_names

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes: {list(target_names)}")
print(f"Features: {feature_names}")

## 2. PCA: Unsupervised Dimensionality Reduction

Following the original R analysis, we log-transform and standardise the features before applying PCA.

In [ ]:
# Log-transform then standardise (faithful to R code: log(iris[,1:4]), center=TRUE, scale.=TRUE)
X_log = np.log(X)
scaler = StandardScaler()
X_log_scaled = scaler.fit_transform(X_log)

# Fit PCA
pca = PCA()
pca.fit(X_log_scaled)

# Manual projection onto first 2 PCs (mirrors R code: scale(log.ir) %*% ir.pca$rotation[,1:2])
X_pca = X_log_scaled @ pca.components_[:2].T

print("Explained variance ratios:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {v:.4f} ({v*100:.1f}%)")
print(f"\nFirst 2 PCs: {sum(pca.explained_variance_ratio_[:2])*100:.1f}%")

## 3. LDA: Supervised Dimensionality Reduction

LDA uses the class labels to find directions that maximise between-class separation. Following the original R code, we use raw features (no log, no scaling).

In [ ]:
# LDA on raw features (faithful to R code: lda(Species ~ ., data = iris))
lda = LinearDiscriminantAnalysis()
lda.fit(X, y)

# Manual projection (mirrors R code: as.matrix(iris[,1:4]) %*% r$scaling[,1:2])
X_lda = X @ lda.scalings_[:, :2]

print("Between-class variance explained:")
for i, v in enumerate(lda.explained_variance_ratio_):
    print(f"  LD{i+1}: {v:.4f} ({v*100:.1f}%)")

## 4. Side-by-Side Comparison: PCA vs LDA

In [ ]:
colours = ['#2196F3', '#FF9800', '#4CAF50']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

for i, name in enumerate(target_names):
    mask = y == i
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colours[i], label=name.capitalize(),
                alpha=0.7, edgecolors='white', linewidth=0.5, s=50)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax1.set_title('PCA Projection (Unsupervised)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

for i, name in enumerate(target_names):
    mask = y == i
    ax2.scatter(X_lda[mask, 0], X_lda[mask, 1], c=colours[i], label=name.capitalize(),
                alpha=0.7, edgecolors='white', linewidth=0.5, s=50)
ax2.set_xlabel(f'LD1 ({lda.explained_variance_ratio_[0]*100:.1f}% between-class variance)')
ax2.set_ylabel(f'LD2 ({lda.explained_variance_ratio_[1]*100:.1f}% between-class variance)')
ax2.set_title('LDA Projection (Supervised)', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Scree Plot and Discriminant Importance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PCA scree plot
pcs = range(1, len(pca.explained_variance_ratio_) + 1)
ax1.bar(pcs, pca.explained_variance_ratio_ * 100, color='#2196F3', alpha=0.7, edgecolor='#1565C0')
ax1.plot(pcs, np.cumsum(pca.explained_variance_ratio_) * 100, 'ro-', markersize=8, linewidth=2)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance (%)')
ax1.set_title('PCA: Scree Plot', fontweight='bold')
ax1.set_xticks(list(pcs))
ax1.set_xticklabels([f'PC{i}' for i in pcs])
ax1.grid(True, alpha=0.3, axis='y')

# LDA singular value proportions
lds = range(1, len(lda.explained_variance_ratio_) + 1)
ax2.bar(lds, lda.explained_variance_ratio_ * 100, color='#FF9800', alpha=0.7, edgecolor='#E65100')
ax2.set_xlabel('Linear Discriminant')
ax2.set_ylabel('Between-Class Variance (%)')
ax2.set_title('LDA: Discriminant Importance', fontweight='bold')
ax2.set_xticks(list(lds))
ax2.set_xticklabels([f'LD{i}' for i in lds])
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. PCA Loadings Biplot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for i, name in enumerate(target_names):
    mask = y == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colours[i], label=name.capitalize(),
               alpha=0.4, s=30, edgecolors='none')

scale_factor = 3.0
short_names = ['Sepal L', 'Sepal W', 'Petal L', 'Petal W']
for j in range(4):
    ax.annotate('',
                xy=(pca.components_[0, j] * scale_factor, pca.components_[1, j] * scale_factor),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.text(pca.components_[0, j] * scale_factor * 1.12,
            pca.components_[1, j] * scale_factor * 1.12,
            short_names[j], fontsize=11, color='red', fontweight='bold',
            ha='center', va='center')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA Loadings Biplot', fontweight='bold')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='grey', linewidth=0.5)
ax.axvline(0, color='grey', linewidth=0.5)

plt.tight_layout()
plt.show()

## 7. The Adversarial Example: When PCA Fails

A synthetic 2D dataset where the direction of maximum variance is perpendicular to the class boundary.

In [ ]:
np.random.seed(42)
n = 150
X_0 = np.column_stack([np.random.randn(n) * 3, np.random.randn(n) * 0.5 - 1])
X_1 = np.column_stack([np.random.randn(n) * 3, np.random.randn(n) * 0.5 + 1])
X_adv = np.vstack([X_0, X_1])
y_adv = np.array([0]*n + [1]*n)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_0[:, 0], X_0[:, 1], c='#2196F3', alpha=0.6, label='Class 0', s=40)
ax.scatter(X_1[:, 0], X_1[:, 1], c='#FF9800', alpha=0.6, label='Class 1', s=40)

# Show PCA direction (horizontal, along max variance)
ax.annotate('', xy=(5, 0), xytext=(-5, 0),
            arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.text(5.2, 0.3, 'PC1\n(max variance)', fontsize=11, color='red', fontweight='bold')

# Show LDA direction (vertical, class separation)
ax.annotate('', xy=(0, 2.5), xytext=(0, -2.5),
            arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
ax.text(0.3, 2.7, 'LD1\n(max separation)', fontsize=11, color='green', fontweight='bold')

ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('The Adversarial Example: Max Variance ⊥ Class Boundary', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# PCA and LDA projections on the adversarial data
pca_adv = PCA(n_components=1)
X_adv_pca = pca_adv.fit_transform(X_adv)

lda_adv = LinearDiscriminantAnalysis(n_components=1)
X_adv_lda = lda_adv.fit_transform(X_adv, y_adv)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))

for cls, (colour, label) in enumerate(zip(['#2196F3', '#FF9800'], ['Class 0', 'Class 1'])):
    mask = y_adv == cls
    ax1.scatter(X_adv_pca[mask, 0], np.random.randn(mask.sum())*0.05,
                c=colour, alpha=0.5, s=30, label=label)
ax1.set_xlabel('PC1 projection')
ax1.set_title('PCA: Complete Overlap (Useless)', fontweight='bold', color='red')
ax1.set_yticks([])
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3, axis='x')

for cls, (colour, label) in enumerate(zip(['#2196F3', '#FF9800'], ['Class 0', 'Class 1'])):
    mask = y_adv == cls
    ax2.scatter(X_adv_lda[mask, 0], np.random.randn(mask.sum())*0.05,
                c=colour, alpha=0.5, s=30, label=label)
ax2.set_xlabel('LD1 projection')
ax2.set_title('LDA: Clean Separation', fontweight='bold', color='green')
ax2.set_yticks([])
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 8. k-NN Accuracy: PCA vs LDA Features

In [ ]:
# Full PCA and LDA transformations on Iris
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_log_scaled)

lda_full = LinearDiscriminantAnalysis()
X_lda_full = lda_full.fit_transform(X, y)

n_comps = [1, 2, 3, 4]
pca_accs = []
lda_accs = []
knn = KNeighborsClassifier(n_neighbors=5)

for k in n_comps:
    pca_score = cross_val_score(knn, X_pca_full[:, :k], y, cv=5, scoring='accuracy').mean()
    pca_accs.append(pca_score)

    k_lda = min(k, X_lda_full.shape[1])
    lda_score = cross_val_score(knn, X_lda_full[:, :k_lda], y, cv=5, scoring='accuracy').mean()
    lda_accs.append(lda_score)

print("k-NN accuracy by number of components:")
for k, pa, la in zip(n_comps, pca_accs, lda_accs):
    print(f"  {k} component(s): PCA={pa:.3f}, LDA={la:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(n_comps))
width = 0.35
ax.bar(x - width/2, [a*100 for a in pca_accs], width, label='PCA + k-NN',
       color='#2196F3', alpha=0.8, edgecolor='#1565C0')
ax.bar(x + width/2, [a*100 for a in lda_accs], width, label='LDA + k-NN',
       color='#FF9800', alpha=0.8, edgecolor='#E65100')

ax.set_xlabel('Number of Components')
ax.set_ylabel('5-Fold CV Accuracy (%)')
ax.set_title('k-NN Classification Accuracy: PCA vs LDA Features', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(n_comps)
ax.legend()
ax.set_ylim(85, 102)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Exercises

1. **Wine dataset:** Replace Iris with `load_wine()` (13 features, 3 classes). Does the PCA vs LDA gap widen or narrow with more features?

2. **Kernel PCA:** Try `KernelPCA(kernel='rbf')` on the adversarial example. Does non-linear PCA recover the class boundary?

3. **Vary n_components:** For the Wine dataset, plot k-NN accuracy vs number of components (1 to 13) for both PCA and LDA. At what point does PCA catch up to LDA?

4. **Regularised LDA:** Generate a high-dimensional dataset with `make_classification(n_features=200, n_samples=50)`. Try standard LDA (will it fail?) and then `LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')`.

5. **QDA comparison:** Fit `QuadraticDiscriminantAnalysis` on Iris and compare its decision boundaries with LDA's linear boundaries. When might QDA outperform LDA?